# Native vs pyTDI Kappa Comparison

This notebook compares two ways of producing the Sagnac data vector used by coronagraphic kappa:

1. **Native/static-link kappa**: current implementation builds first-generation static Sagnac `alpha,beta,gamma` directly from the six link CSVs.
2. **pyTDI-Sagnac kappa**: pyTDI builds `ALPHA/BETA/GAMMA`, then the same static coronagraphic projection `A(f, sky)` is applied.

This is a convention test. Agreement is not guaranteed until delay/Fourier/sign/scaling conventions are matched.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts").exists() and (PROJECT_ROOT.parent / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

CONFIG = PROJECT_ROOT / "test-run" / "config.yaml"
CATALOG = PROJECT_ROOT / "test-run" / "catalog.csv"
RUNS_ROOT = PROJECT_ROOT / "test-run" / "simulations"
SOURCE_ID = "smbhb_m1e6"
SOURCE = "noisy"  # change to "gw_only" if desired
MAX_SAMPLES = 65536
OUT_STD = PROJECT_ROOT / "test-run" / "native_vs_pytdi_kappa" / "standard_sagnac"
OUT_ETA = PROJECT_ROOT / "test-run" / "native_vs_pytdi_kappa" / "eta_sagnac"
RUN_ETA = False  # set True only if your pyTDI input file contains eta_ij measurements

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"standard output = {OUT_STD}")
print(f"eta output      = {OUT_ETA}")

## 1. Run Standard pyTDI Sagnac Comparison

This uses pyTDI `ALPHA1/BETA1/GAMMA1` and compares against the native static-link Sagnac kappa on the same centered chunk.

In [ ]:
!conda run -n lisa-tf python scripts/pytdi_sagnac_kappa_example.py \
  --config {CONFIG} \
  --catalog {CATALOG} \
  --runs-root {RUNS_ROOT} \
  --source-id {SOURCE_ID} \
  --source {SOURCE} \
  --generation 1 \
  --max-samples {MAX_SAMPLES} \
  --output-dir {OUT_STD}

## 2. Optional: pyTDI ETA Sagnac Comparison

pyTDI `ALPHA1_ETA/BETA1_ETA/GAMMA1_ETA` requires `eta_ij` measurements in the input file. The current `lisainstrument` product has `sci_ij/ref_ij/tmi_ij`, not `eta_ij`, so this comparison is skipped by default.

Set `RUN_ETA = True` in the setup cell only if your input file contains `eta_12, eta_23, eta_31, eta_13, eta_32, eta_21`.

In [ ]:
if not RUN_ETA:
    print("Skipping ETA Sagnac comparison: current measurement file does not contain eta_ij channels.")
else:
    import subprocess

    eta_cmd = [
        "conda", "run", "-n", "lisa-tf", "python", "scripts/pytdi_sagnac_kappa_example.py",
        "--config", str(CONFIG),
        "--catalog", str(CATALOG),
        "--runs-root", str(RUNS_ROOT),
        "--source-id", SOURCE_ID,
        "--source", SOURCE,
        "--generation", "1",
        "--eta-combinations",
        "--max-samples", str(MAX_SAMPLES),
        "--output-dir", str(OUT_ETA),
    ]
    eta_result = subprocess.run(eta_cmd, text=True, capture_output=True)
    print(eta_result.stdout)
    if eta_result.returncode != 0:
        print("ETA Sagnac comparison did not run. This usually means the input measurement file lacks eta_ij channels.")
        print(eta_result.stderr)
    else:
        print("ETA Sagnac comparison completed.")

## 3. Load Summaries

In [ ]:
def load_summary(out_dir):
    path = out_dir / "summary.json"
    if not path.exists():
        return None
    return json.loads(path.read_text())

rows = []
std_summary = load_summary(OUT_STD)
eta_summary = load_summary(OUT_ETA)
if std_summary is not None:
    rows.append({"case": "pyTDI ALPHA1/BETA1/GAMMA1", **std_summary})
if eta_summary is not None:
    rows.append({"case": "pyTDI ALPHA1_ETA/BETA1_ETA/GAMMA1_ETA", **eta_summary})
else:
    rows.append({
        "case": "pyTDI ALPHA1_ETA/BETA1_ETA/GAMMA1_ETA",
        "source_id": SOURCE_ID,
        "source": SOURCE,
        "pytdi_sagnac_generation": 1,
        "pytdi_eta_combinations": True,
        "n_samples": None,
        "dt_s": None,
        "pytdi_static_real_correlation": None,
        "pytdi_rms": None,
        "static_rms": None,
    })

summaries = pd.DataFrame(rows)
cols = [
    "case", "source_id", "source", "pytdi_sagnac_generation", "pytdi_eta_combinations",
    "n_samples", "dt_s", "pytdi_static_real_correlation", "pytdi_rms", "static_rms",
]
summaries[cols]

## 4. Direct Time-Series Comparison

The CSV contains the two kappa time series evaluated at the injected sky direction:

- `kappa_pytdi_real`: kappa using pyTDI Sagnac variables.
- `kappa_static_real`: kappa using the native static-link Sagnac variables.

The native and pyTDI amplitudes can differ strongly before convention normalization, so the second plot z-scores both series.

In [ ]:
def load_timeseries(out_dir):
    path = out_dir / "pytdi_sagnac_kappa_timeseries.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df["time_offset_s"] = df["t_waveform"] - df["t_waveform"].iloc[0]
    return df

def zscore(x):
    x = np.asarray(x, dtype=float)
    return (x - np.nanmean(x)) / np.nanstd(x)

std = load_timeseries(OUT_STD)
eta = load_timeseries(OUT_ETA)
if std is None:
    raise FileNotFoundError(f"Missing standard comparison CSV under {OUT_STD}")

fig, axes = plt.subplots(2, 2, figsize=(14, 7), constrained_layout=True, sharex='col')
plot_cases = [(axes[0,0], axes[1,0], std, 'standard Sagnac')]
if eta is not None:
    plot_cases.append((axes[0,1], axes[1,1], eta, 'ETA Sagnac'))
else:
    axes[0,1].axis('off')
    axes[1,1].axis('off')
    axes[0,1].text(0.05, 0.5, 'ETA Sagnac not available\nmissing eta_ij measurements', fontsize=12)

for raw_ax, z_ax, df, title in plot_cases:
    raw_ax.plot(df['time_offset_s'], df['kappa_pytdi_real'], lw=0.8, label='pyTDI kappa')
    raw_ax.plot(df['time_offset_s'], df['kappa_static_real'], lw=0.8, alpha=0.7, label='native/static kappa')
    raw_ax.set_title(title)
    raw_ax.set_ylabel('raw kappa')
    raw_ax.legend(loc='best')
    raw_ax.grid(alpha=0.25)

    z_ax.plot(df['time_offset_s'], zscore(df['kappa_pytdi_real']), lw=0.8, label='pyTDI kappa z')
    z_ax.plot(df['time_offset_s'], zscore(df['kappa_static_real']), lw=0.8, alpha=0.7, label='native/static kappa z')
    z_ax.set_title(f'{title}, z-scored')
    z_ax.set_xlabel('time since chunk start [s]')
    z_ax.set_ylabel('z-score')
    z_ax.legend(loc='best')
    z_ax.grid(alpha=0.25)
plt.show()

## 5. Correlation With Sign And Scale Fits

This fits a simple scalar relation `native ≈ a * pyTDI + b`. If the only issue is scale/sign/offset, correlation should still be high and the fitted residual should be small. If correlation is near zero, the two constructions are not convention-compatible yet.

In [ ]:
def fit_summary(df):
    x = df['kappa_pytdi_real'].to_numpy(dtype=float)
    y = df['kappa_static_real'].to_numpy(dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]
    a, b = np.polyfit(x, y, deg=1)
    yhat = a * x + b
    corr = np.corrcoef(x, y)[0, 1]
    rms_y = np.sqrt(np.mean(y**2))
    rms_resid = np.sqrt(np.mean((y - yhat)**2))
    return {
        'slope_a': a,
        'offset_b': b,
        'corr': corr,
        'native_rms': rms_y,
        'fit_residual_rms': rms_resid,
        'residual_over_native_rms': rms_resid / rms_y,
    }

fit_rows = [{'case': 'standard', **fit_summary(std)}]
if eta is not None:
    fit_rows.append({'case': 'eta', **fit_summary(eta)})
fit_table = pd.DataFrame(fit_rows)
fit_table

## 6. View Script-Generated Plots

In [ ]:
for label, out_dir in [('standard', OUT_STD), ('eta', OUT_ETA)]:
    print(label)
    if not (out_dir / 'summary.json').exists():
        print(f'Skipping {label}: no summary found at {out_dir}')
        continue
    for name in ['plots/pytdi_vs_static_kappa_time.png', 'plots/pytdi_kappa_spectrum.png']:
        path = out_dir / name
        if path.exists():
            display(Image(filename=str(path)))
        else:
            print(f'Missing {path}')

## Interpretation

Use this decision rule:

- If standard or ETA correlation is high after a possible sign flip, pyTDI Sagnac is likely usable after scaling/sign convention fixes.
- If both correlations are near zero, the pyTDI observable basis or delay/Fourier convention is not aligned with the current static kappa projection.
- If pyTDI generation 2 is desired, the kappa response vector `A(f, sky)` should be derived for the matching second-generation Sagnac observable, not reused blindly from the static first-generation formula.